# GUI-Model: Mobile UI Transition & Action Predictor

Qwen3-VL-8B-Instruct 기반 모바일 UI 예측 모델 Fine-tuning 파이프라인.

**전체 파이프라인:**
1. **Stage 1: World Modeling** - UI State (XML) + Action → Next UI State (XML)
2. **Stage 2: Action Prediction** - Screenshot + UI State + Task → Action (JSON)
3. **Merge & Upload** - Merge 후 HuggingFace Hub 업로드

**모델 정보:**

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Template | qwen3_vl_nothink |
| Vision Tower | Frozen |

**데이터셋 정보:**

| Dataset | Stage | Entries | Description |
|---------|-------|---------|-------------|
| GUI-Model_stage1_train | 1 | 3,087 | 전체 (OpenApp 포함) |
| GUI-Model_stage1_NoOpenApp_train | 1 | 2,073 | OpenApp 제외 |
| GUI-Model_stage2_train | 2 | 3,655 | Action Prediction |
| Images | - | 3,655 | Mobile UI Screenshots (PNG, 공유) |

**Stage 1 학습 실험:**

| ID | Method | LR Schedule | Learning Rate | Config |
|----|--------|-------------|---------------|--------|
| [1] | Full FT | constant | 2e-5 | stage1_full/qwen3_vl_8b_gui.yaml |

**Stage 2 학습 실험 (3-Way 비교):**

| ID | Base Model | HuggingFace ID | Config |
|----|------------|----------------|--------|
| [S2-1] | Qwen3-VL-8B (Baseline) | `Qwen/Qwen3-VL-8B-Instruct` | stage2_lora/baseline.yaml |
| [S2-2] | Code2World-8B | `GD-ML/Code2World` | stage2_lora/code2world.yaml |
| [S2-3] | gWorld-8B | `trillionlabs/gWorld-8B` | stage2_lora/gworld.yaml |

**Stage 2 공통 설정:** LoRA r=16, α=32, dropout=0.1, LR=5e-5 (cosine), batch=4x2x2GPU=16, A100x2

## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive/')

In [ ]:
import os

# ============================================================
# === 여기서 프로젝트 루트 경로를 설정하세요 ===
BASE_DIR = "/path/to/GUI-Model"
# ============================================================

LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")
DATA_DIR = os.path.join(BASE_DIR, "data")

os.chdir(BASE_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/hiyouga/LlamaFactory.git

In [ ]:
os.chdir(LF_ROOT)
!pip install -e ".[torch,metrics]"

## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 LLaMA-Factory에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: 3,655개 모바일 UI 스크린샷

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage1.jsonl | 3,087 | 전체 (OpenApp 포함) |
| gui-model_stage1_NoOpenApp.jsonl | 2,073 | OpenApp 액션 제외 |

In [ ]:
import json
import shutil
from pathlib import Path

# === Paths ===
DATA_PATH = Path(DATA_DIR)
LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. 데이터 디렉토리 생성 ===
LF_GUI_DATA.mkdir(parents=True, exist_ok=True)
(LF_GUI_DATA / "images").mkdir(exist_ok=True)

# === 2. JSONL 파일 복사 ===
for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_NoOpenApp.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f"[Copy] {fname} ({src.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"[WARN] Not found: {src}")

# === 3. 이미지 복사 ===
src_images = DATA_PATH / "images"
dst_images = LF_GUI_DATA / "images"

if src_images.exists():
    img_count = len(list(src_images.glob("*.png")))
    existing = len(list(dst_images.glob("*.png")))
    if existing >= img_count:
        print(f"[Skip] Images already exist ({existing} files)")
    else:
        for img in src_images.glob("*.png"):
            dst_img = dst_images / img.name
            if not dst_img.exists():
                shutil.copy2(img, dst_img)
        print(f"[Copy] {img_count} images")
else:
    print(f"[WARN] Image directory not found: {src_images}")

# === 4. dataset_info.json 등록 ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

DATASETS_TO_REGISTER = {
    "GUI-Model_stage1_train": {
        "file_name": "GUI-Model/gui-model_stage1.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
            "system_tag": "system"
        }
    },
    "GUI-Model_stage1_NoOpenApp_train": {
        "file_name": "GUI-Model/gui-model_stage1_NoOpenApp.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
            "system_tag": "system"
        }
    }
}

for name, config in DATASETS_TO_REGISTER.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 1 Dataset Statistics ===\n")

for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_NoOpenApp.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()
        entry_count = len(lines)

        # 첫 번째 엔트리 샘플
        sample = json.loads(lines[0])
        msg_count = len(sample["messages"])
        img_count = len(sample.get("images", []))

        print(f"[{fname}]")
        print(f"  Entries: {entry_count}")
        print(f"  Sample - Messages: {msg_count}, Images: {img_count}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
        print()

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 1.5 Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 변환하고 LLaMA-Factory에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1과 동일한 3,655개 (공유)

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage2.jsonl | 3,655 | Action Prediction (전체) |

**변환 작업:**
- 원본의 `id`, `assets`, `source` 필드 제거
- `images` 필드 추가: `["GUI-Model/images/{id}.png"]`

In [ ]:
import json
from pathlib import Path

stage2_path = Path(DATA_DIR) / "gui-model_stage2.jsonl"

# === 1. 변환 필요 여부 확인 ===
with open(stage2_path, 'r') as f:
    first = json.loads(f.readline())

if "images" in first and "id" not in first:
    print("[Skip] Stage 2 data already converted")
    print(f"  Keys: {list(first.keys())}")
    print(f"  Images: {first['images']}")
else:
    # === 2. 변환 수행 ===
    with open(stage2_path, 'r') as f:
        lines = f.readlines()

    print(f"[Before] {len(lines)} entries, Keys: {list(first.keys())}")

    converted = []
    for line in lines:
        entry = json.loads(line)
        new_entry = {
            "messages": entry["messages"],
            "images": [f"GUI-Model/images/{entry['id']}.png"]
        }
        converted.append(json.dumps(new_entry, ensure_ascii=False))

    with open(stage2_path, 'w') as f:
        f.write('\n'.join(converted) + '\n')

    # === 3. 검증 ===
    with open(stage2_path, 'r') as f:
        check = json.loads(f.readline())

    print(f"[After] {len(converted)} entries, Keys: {list(check.keys())}")
    print(f"  Sample images: {check['images']}")
    print(f"  File size: {stage2_path.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
import json
import shutil
from pathlib import Path

DATA_PATH = Path(DATA_DIR)
LF_DATA_DIR = Path(LF_ROOT) / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. Stage 2 JSONL 복사 ===
src = DATA_PATH / "gui-model_stage2.jsonl"
dst = LF_GUI_DATA / "gui-model_stage2.jsonl"
if src.exists():
    shutil.copy2(src, dst)
    print(f"[Copy] gui-model_stage2.jsonl ({src.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print(f"[WARN] Not found: {src}")

# === 2. dataset_info.json에 Stage 2 등록 ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"
with open(dataset_info_path, 'r', encoding='utf-8') as f:
    dataset_info = json.load(f)

dataset_info["GUI-Model_stage2_train"] = {
    "file_name": "GUI-Model/gui-model_stage2.jsonl",
    "formatting": "sharegpt",
    "columns": {"messages": "messages", "images": "images"},
    "tags": {
        "role_tag": "from",
        "content_tag": "value",
        "user_tag": "human",
        "assistant_tag": "gpt",
        "system_tag": "system"
    }
}

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"[Registered] GUI-Model_stage2_train")
print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"
fpath = LF_GUI_DATA / "gui-model_stage2.jsonl"

print("=== Stage 2 Dataset Statistics ===\n")

if fpath.exists():
    with open(fpath, 'r') as f:
        lines = f.readlines()

    print(f"[gui-model_stage2.jsonl]")
    print(f"  Entries: {len(lines)}")
    print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

    # Action type 분포
    action_types = Counter()
    for line in lines:
        entry = json.loads(line)
        gpt_val = json.loads(entry['messages'][-1]['value'])
        action_types[gpt_val.get('type', 'unknown')] += 1

    print(f"\n  Action Type Distribution:")
    for atype, count in action_types.most_common():
        print(f"    {atype}: {count} ({count/len(lines)*100:.1f}%)")

    # 샘플 확인
    sample = json.loads(lines[0])
    print(f"\n  Sample entry:")
    print(f"    Keys: {list(sample.keys())}")
    print(f"    Messages: {len(sample['messages'])} (roles: {[m['from'] for m in sample['messages']]})")
    print(f"    Images: {sample['images']}")
else:
    print(f"[WARN] Not found: {fpath}")

## 2. Stage 1 SFT Training

Qwen3-VL-8B-Instruct를 Stage 1 (World Modeling) 데이터로 Full Fine-tuning합니다.
DeepSpeed ZeRO-3 필수 (2x A100 이상 권장).

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Method | Full (all parameters) |
| Dataset | GUI-Model_stage1_NoOpenApp_train (2,073건) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 8,192 |
| Epochs | 3.0 |
| Learning Rate | 2e-5 |
| LR Scheduler | constant |
| Warmup | 0.0 |
| Effective Batch | 16 (1 x 8 x 2 GPU) |
| Precision | bf16 |
| Vision Tower | Frozen |
| DeepSpeed | ZeRO-3 |
| Gradient Checkpointing | Enabled |
| Weight Decay | 0.01 |
| Hardware | A100 80GB x 2 |
| Save Steps | 50 |
| Eval Steps | 50 |

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage1_full"
yaml_dir.mkdir(parents=True, exist_ok=True)

STAGE1_CONFIG = """\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4194304

### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_NoOpenApp_train
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_full
logging_steps: 1
save_steps: 50
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-5
num_train_epochs: 3.0
lr_scheduler_type: constant
warmup_ratio: 0.0
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
deepspeed: examples/deepspeed/ds_z3_config.json

# ### eval
# val_size: 0.1
# per_device_eval_batch_size: 1
# eval_strategy: steps
# eval_steps: 50
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## [1] Full Fine-tuning (constant, lr=2e-5, DeepSpeed Z3)
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage1_full/qwen3_vl_8b_gui.yaml

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "gui"
yaml_dir.mkdir(parents=True, exist_ok=True)

MERGE_STAGE1_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full
trust_remote_code: true
template: qwen3_vl_nothink

### export
export_dir: ./exports/qwen3-vl-8b-gui
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: SaFD-00/qwen3-vl-8b-gui
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(MERGE_STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

## 3. Stage 1 Merge & Upload to HuggingFace

학습된 Stage 1 LoRA 어댑터를 기본 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**업로드 대상:** `SaFD-00/qwen3-vl-8b-gui`

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
os.chdir(LF_ROOT)

## LoRA Merge & HuggingFace Hub Upload
!llamafactory-cli export examples/merge_custom/GUI-Model/gui/qwen3_vl_8b_gui.yaml

## 4. Stage 2 SFT Training - World Model 사전학습 효과 검증

**핵심 가설**: GUI World Modeling으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT에서 더 빠르게 수렴할 것이다.

### 3-Way 비교 설계

| ID | 역할 | 모델 | HuggingFace ID |
|----|------|------|----------------|
| [S2-1] | Baseline | Qwen3-VL-8B-Instruct | `Qwen/Qwen3-VL-8B-Instruct` |
| [S2-2] | World Model A | Code2World-8B | `GD-ML/Code2World` |
| [S2-3] | World Model B | gWorld-8B | `trillionlabs/gWorld-8B` |

> 3개 모두 동일한 베이스 모델(Qwen3-VL-8B-Instruct)에서 파생. 공정한 비교 가능.

**공통 학습 설정:**

| 항목 | 값 |
|------|-----|
| Dataset | GUI-Model_stage2_train (3,655건) |
| Method | LoRA (rank=16, alpha=32, target=all, dropout=0.1) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 4,096 |
| Epochs | 3.0 |
| Learning Rate | 5e-5 (cosine, warmup=0.05) |
| Effective Batch | 16 (4 x 2 x 2 GPU) |
| Weight Decay | 0.01 |
| Val Split | 0.1 |
| Eval Steps | 50 |
| Logging Steps | 1 |
| Precision | bf16 |
| Vision Tower | Frozen |
| Hardware | A100 80GB x 2 |

In [ ]:
import os
from pathlib import Path

# === Stage 2 YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage2_lora"
yaml_dir.mkdir(parents=True, exist_ok=True)

# 공통 설정 (model_name_or_path, output_dir만 다름)
# 연구 기반 하이퍼파라미터 (Qwen-GUI-3B, ShowUI, LR Matters 논문 참고)
COMMON_CONFIG = """\
### model
model_name_or_path: {model_name_or_path}
trust_remote_code: true
image_max_pixels: 4194304

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: true
lora_rank: 16
lora_alpha: 32
lora_target: all
lora_dropout: 0.1

### dataset
dataset: GUI-Model_stage2_train
template: qwen3_vl_nothink
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{output_dir}
logging_steps: 1
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 2
learning_rate: 5.0e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
weight_decay: 0.01
bf16: true
gradient_checkpointing: true

# ### eval
# val_size: 0.1
# per_device_eval_batch_size: 4
# eval_strategy: steps
# eval_steps: 50
# compute_accuracy: true
"""

MODELS = {
    "baseline.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "output_dir": "stage2_baseline",
    },
    "code2world.yaml": {
        "model_name_or_path": "GD-ML/Code2World",
        "output_dir": "stage2_code2world",
    },
    "gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "output_dir": "stage2_gworld",
    },
}

for fname, params in MODELS.items():
    content = COMMON_CONFIG.format(**params)
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  model: {params['model_name_or_path']}")
    print(f"  output: ./outputs/{params['output_dir']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [S2-1] Stage 2 - Baseline (Qwen3-VL-8B-Instruct) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/baseline.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-2] Stage 2 - Code2World-8B (World Model A) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/code2world.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-3] Stage 2 - gWorld-8B (World Model B) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/gworld.yaml

## 6. Stage 2 Merge & Upload

Stage 2에서 최적의 결과를 보인 모델의 LoRA 어댑터를 병합하고 업로드합니다.

> 학습 결과를 비교한 후, 아래 merge YAML의 `adapter_name_or_path`를 해당 모델의 checkpoint 경로로 수정하세요.

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "stage2"
yaml_dir.mkdir(parents=True, exist_ok=True)

# === 최적 모델 설정 (결과 비교 후 수정) ===
BEST_BASE_MODEL = "Qwen/Qwen3-VL-8B-Instruct"  # 또는 GD-ML/Code2World, trillionlabs/gWorld-8B
BEST_ADAPTER_PATH = "./outputs/stage2_baseline"   # 또는 stage2_code2world, stage2_gworld
EXPORT_HUB_ID = "SaFD-00/qwen3-vl-8b-gui-stage2"

MERGE_STAGE2_CONFIG = f"""\
### model
model_name_or_path: {BEST_BASE_MODEL}
adapter_name_or_path: {BEST_ADAPTER_PATH}
trust_remote_code: true
template: qwen3_vl_nothink
finetuning_type: lora

### export
export_dir: ./exports/stage2_best
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {EXPORT_HUB_ID}
"""

fpath = yaml_dir / "best_model.yaml"
with open(fpath, 'w') as f:
    f.write(MERGE_STAGE2_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
print(f"  Base: {BEST_BASE_MODEL}")
print(f"  Adapter: {BEST_ADAPTER_PATH}")

In [ ]:
os.chdir(LF_ROOT)

## Stage 2 Best Model - LoRA Merge & Upload
## 위 셀에서 BEST_BASE_MODEL, BEST_ADAPTER_PATH를 최적 모델로 수정 후 실행하세요
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/best_model.yaml